In [2]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from typing import List, Dict

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("Setup complete!")

Setup complete!


In [3]:
podcast_text = """
Trustworthy AI means building systems that are safe, fair, and accountable.
Transparency in AI is essential for public trust and adoption.
Bias in AI systems can cause harm to vulnerable populations.
Human oversight is critical when deploying AI in high-stakes domains.
Explainability helps users understand why AI made a specific decision.
Responsible AI development requires diverse teams and inclusive datasets.
"""

eu_ai_act_text = """
The EU AI Act establishes a risk-based framework for artificial intelligence.
High-risk AI systems must meet strict requirements before market placement.
Prohibited AI practices include social scoring and real-time biometric surveillance.
Providers of general-purpose AI models must maintain technical documentation.
The Act applies to providers and deployers of AI systems in the European Union.
Conformity assessments are required for high-risk AI systems.
"""

print("Documents loaded!")
print(f"Podcast text length: {len(podcast_text)} chars")
print(f"EU AI Act text length: {len(eu_ai_act_text)} chars")

Documents loaded!
Podcast text length: 416 chars
EU AI Act text length: 460 chars


In [4]:
def chunk_with_metadata(text: str, source: str, chunk_size: int = 300, overlap: int = 50) -> List[Dict]:
    chunks = []
    start = 0
    chunk_id = 0
    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end].strip()
        if chunk_text:
            chunks.append({
                "text": chunk_text,
                "metadata": {"source": source, "chunk_id": chunk_id}
            })
            chunk_id += 1
        start += chunk_size - overlap
    return chunks

podcast_chunks = chunk_with_metadata(podcast_text, source="podcast")
eu_act_chunks = chunk_with_metadata(eu_ai_act_text, source="eu_ai_act")
all_chunks = podcast_chunks + eu_act_chunks

print(f"Podcast chunks: {len(podcast_chunks)}")
print(f"EU AI Act chunks: {len(eu_act_chunks)}")
print(f"Total chunks: {len(all_chunks)}")

Podcast chunks: 2
EU AI Act chunks: 2
Total chunks: 4


In [5]:
def get_embeddings_batch(texts: List[str], model: str = "text-embedding-3-small") -> List[List[float]]:
    all_embeddings = []
    batch_size = 100
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        response = client.embeddings.create(model=model, input=batch)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
        print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} chunks")
    return all_embeddings

texts = [chunk["text"] for chunk in all_chunks]
embeddings = get_embeddings_batch(texts)
print(f"\nGenerated {len(embeddings)} embeddings!")

Processed 4/4 chunks

Generated 4 embeddings!


In [6]:
def cosine_similarity(vec1: List[float], vec2: List[float]) -> float:
    v1 = np.array(vec1)
    v2 = np.array(vec2)
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def basic_search(query: str, chunks: List[Dict], chunk_embeddings: List[List[float]], top_k: int = 5, source_filter: str = None) -> List[Dict]:
    query_embedding = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    ).data[0].embedding
    results = []
    for i, (chunk, emb) in enumerate(zip(chunks, chunk_embeddings)):
        if source_filter and chunk["metadata"]["source"] != source_filter:
            continue
        score = cosine_similarity(query_embedding, emb)
        results.append({
            "text": chunk["text"],
            "metadata": chunk["metadata"],
            "similarity_score": round(score, 4)
        })
    results.sort(key=lambda x: x["similarity_score"], reverse=True)
    return results[:top_k]

query = "What are the requirements for high-risk AI systems?"
print(f"Query: {query}\n")
print("=== RESULTS WITHOUT RERANKING ===\n")
basic_results = basic_search(query, all_chunks, embeddings, top_k=5)
for i, result in enumerate(basic_results):
    print(f"Result {i+1} | Source: {result['metadata']['source']} | Score: {result['similarity_score']}")
    print(f"Text: {result['text'][:150]}...")
    print()

Query: What are the requirements for high-risk AI systems?

=== RESULTS WITHOUT RERANKING ===

Result 1 | Source: eu_ai_act | Score: 0.6937
Text: The EU AI Act establishes a risk-based framework for artificial intelligence.
High-risk AI systems must meet strict requirements before market placeme...

Result 2 | Source: eu_ai_act | Score: 0.6325
Text: of general-purpose AI models must maintain technical documentation.
The Act applies to providers and deployers of AI systems in the European Union.
Co...

Result 3 | Source: podcast | Score: 0.5334
Text: high-stakes domains.
Explainability helps users understand why AI made a specific decision.
Responsible AI development requires diverse teams and incl...

Result 4 | Source: podcast | Score: 0.5317
Text: Trustworthy AI means building systems that are safe, fair, and accountable.
Transparency in AI is essential for public trust and adoption.
Bias in AI ...



In [7]:
def rerank_with_llm(query: str, results: List[Dict], top_k: int = 3) -> List[Dict]:
    """Rerank results using LLM relevance scoring"""
    scored_results = []
    for result in results:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "system",
                    "content": "Score how relevant this text is to the query. Reply with ONLY a number from 0 to 10."
                },
                {
                    "role": "user",
                    "content": f"Query: {query}\n\nText: {result['text']}"
                }
            ]
        )
        try:
            score = float(response.choices[0].message.content.strip())
        except:
            score = 0.0
        scored_results.append({**result, "rerank_score": round(score / 10, 4)})

    scored_results.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored_results[:top_k]

print("=== RESULTS WITH LLM RERANKING ===\n")
reranked_results = rerank_with_llm(query, basic_results, top_k=3)
for i, result in enumerate(reranked_results):
    print(f"Result {i+1} | Source: {result['metadata']['source']}")
    print(f"Similarity: {result['similarity_score']} | Rerank Score: {result['rerank_score']}")
    print(f"Text: {result['text'][:150]}...")
    print()

=== RESULTS WITH LLM RERANKING ===

Result 1 | Source: eu_ai_act
Similarity: 0.6937 | Rerank Score: 0.8
Text: The EU AI Act establishes a risk-based framework for artificial intelligence.
High-risk AI systems must meet strict requirements before market placeme...

Result 2 | Source: eu_ai_act
Similarity: 0.6325 | Rerank Score: 0.8
Text: of general-purpose AI models must maintain technical documentation.
The Act applies to providers and deployers of AI systems in the European Union.
Co...

Result 3 | Source: podcast
Similarity: 0.5317 | Rerank Score: 0.7
Text: Trustworthy AI means building systems that are safe, fair, and accountable.
Transparency in AI is essential for public trust and adoption.
Bias in AI ...



In [8]:
def rag_with_reranking(question: str, chunks: List[Dict], chunk_embeddings: List[List[float]], use_reranking: bool = True) -> str:
    initial_results = basic_search(question, chunks, chunk_embeddings, top_k=4)
    if use_reranking:
        final_results = rerank_with_llm(question, initial_results, top_k=3)
        method = "WITH RERANKING"
    else:
        final_results = initial_results[:3]
        method = "WITHOUT RERANKING"
    context = "\n\n".join([f"Source ({r['metadata']['source']}):\n{r['text']}" for r in final_results])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful assistant. Answer based only on the provided context. Cite sources."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    answer = response.choices[0].message.content
    print(f"=== ANSWER {method} ===")
    print(answer)
    print()
    return answer

questions = [
    "What are the requirements for high-risk AI systems?",
    "Why is transparency important in AI?"
]

for question in questions:
    print(f"Question: {question}\n")
    rag_with_reranking(question, all_chunks, embeddings, use_reranking=False)
    rag_with_reranking(question, all_chunks, embeddings, use_reranking=True)
    print("=" * 60 + "\n")

Question: What are the requirements for high-risk AI systems?

=== ANSWER WITHOUT RERANKING ===
High-risk AI systems must meet strict requirements before market placement, including undergoing conformity assessments. Additionally, providers of general-purpose AI models are required to maintain technical documentation related to their systems (Source: eu_ai_act).

=== ANSWER WITH RERANKING ===
High-risk AI systems must meet strict requirements before they can be placed on the market, as established by the EU AI Act. These requirements are part of a risk-based framework that ensures the systems are safe, fair, and accountable. Additionally, conformity assessments are required for these high-risk AI systems to ensure compliance with the established regulations (Source: eu_ai_act).


Question: Why is transparency important in AI?

=== ANSWER WITHOUT RERANKING ===
Transparency in AI is essential for building public trust and promoting adoption of AI technologies. It allows users to understa

In [9]:
print("=== PERFORMANCE COMPARISON ===\n")
print(f"{'Question':<45} {'Without Reranking':<20} {'With Reranking'}")
print("-" * 85)

eval_questions = [
    "What are the requirements for high-risk AI systems?",
    "Why is transparency important in AI?",
    "What is prohibited under the EU AI Act?"
]

for question in eval_questions:
    results_without = basic_search(question, all_chunks, embeddings, top_k=3)
    results_with = rerank_with_llm(question, basic_search(question, all_chunks, embeddings, top_k=4), top_k=3)
    top_source_without = results_without[0]["metadata"]["source"]
    top_source_with = results_with[0]["metadata"]["source"]
    print(f"{question[:44]:<45} {top_source_without:<20} {top_source_with}")

print("\nReranking helps find more precise sources!")

=== PERFORMANCE COMPARISON ===

Question                                      Without Reranking    With Reranking
-------------------------------------------------------------------------------------
What are the requirements for high-risk AI s  eu_ai_act            eu_ai_act
Why is transparency important in AI?          podcast              podcast
What is prohibited under the EU AI Act?       eu_ai_act            eu_ai_act

Reranking helps find more precise sources!
